## Connect this notebook to a Jupyter server running on your HPC

### Instructions to Connect Google Colab to an HPC Jupyter Notebook Server

Follow these steps to run a Jupyter server on your HPC cluster and connect it to Google Colab:

---

#### **1. Start a `tmux` Session**
On your HPC terminal, start a `tmux` session to keep your work running even if you disconnect:
```bash
tmux
```

---

#### **2. Request an Interactive Node**
Request the required resources for your notebook using the `salloc` command:
```bash
salloc -N 1 -n 1 -c 4 -t 4:00:00 --mem=12GB --gres=gpu:tesla-v100:1
```

Once allocated, you will be placed on a compute node (e.g., `node1806`).

---

#### **3. Change to the Desired Directory**
Navigate to the directory containing your data or project files:
```bash
cd ~/data
```

---

#### **4. Activate the Conda/Mamba Environment**
Activate the environment you plan to work in:
```bash
mamba activate worm_env
```

---

#### **5. Launch the Jupyter Server**
Start the Jupyter server with the appropriate settings:
```bash
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser \
    --NotebookApp.password='' \
    --NotebookApp.allow_origin='https://colab.research.google.com' \
    --NotebookApp.port_retries=0
```

This will print a URL with a token, such as:
```
http://127.0.0.1:8888/lab?token=f80cd1e1bf86cb364abb0fa0cf551fe6ae9317b11454bd32
```
Leave this terminal running in the `tmux` session.

---

#### **6. Set Up an SSH Tunnel on Your Local Machine**
On your local computer, create an SSH tunnel to forward the Jupyter server port:
```bash
ssh -f -N -L 8888:node1806:8888 qsimeon@eofe10.mit.edu
```

This will forward the server on `node1806:8888` to `localhost:8888` on your local machine.

---

#### **7. Connect to the Jupyter Server from Google Colab**
1. Open Google Colab and go to **Runtime > Change runtime type > Connect to a local runtime**.
2. In the **Backend URL** field, paste the URL from the Jupyter server output, replacing `127.0.0.1` with `localhost`:
   ```
   http://localhost:8888/lab?token=f80cd1e1bf86cb364abb0fa0cf551fe6ae9317b11454bd32
   ```
3. Click **Connect**.

---

#### **8. Verify the Connection**
- Your Google Colab runtime will now use the Jupyter server running on the HPC node.
- You can access files and leverage the resources of the cluster node directly from Colab.

---

#### **Additional Tips**
- To detach the `tmux` session, press `Ctrl+b`, then `d`.
- To reconnect to the session later, use:
  ```bash
  tmux attach
  ```
- Make sure to terminate the Jupyter server and release the node when done.

These instructions ensure you can seamlessly connect Google Colab to your HPC environment.

In [ ]:
!which python # are you using the Conda env?

~/.conda/envs/worm_env/bin/python


In [ ]:
!pwd # what directory are you in currently

/orcd/home/001/qsimeon/data


In [ ]:
!ls # list files

 20250131_test_worm.tiff
 3dcelltracker_demo_data
 3DeeCellTracker
 behavior_video.npy
'C Elegans Datasets'
 Creamer_LDS_2024
 h5_to_timestamped_nir.ipynb
 jupyter_62783408.err
 jupyter_62783408.out
 jupyter.out
 learn-parallel-hpc
 nd2_to_h5.ipynb
 nd2_to_h5.py
 neural_activitiy.npy
 neuron_ids.npy
 serve_jupyter.sh
 sub-2022-06-14-07-SWF702_ses-20220614_behavior+image+ophys.nwb
 sub-2023-01-05-18-SWF702_ses-20230105_behavior+image+ophys.nwb
 targettrack
 test_worm.nwb
 UCE-clone
 uce-files
 worm1_slow_end.nd2
 worm_env.yml
 worm-graph


### Loading the data from NWB files

We need to get the data we want. There is some calcium data traces associated with NIR videos of behavior. The full dataset of all the worms is a massive 1TB dataset from DANDI. Here is its description:

```
Brain-wide representations of behavior spanning multiple timescales and states in C. elegans
DOI:
10.48324/dandi.000776/0.241009.1509
ID: 000776
0.241009.1509
 Contact Flavell, Steve
 File Count 38
 Size 1 TB
 Created October 9, 2024
 Last update October 9, 2024
 Licenses: spdx:CC-BY-4.0
 Access Information: dandi:OpenAccess
Dataset of 38 worms from 'Brain-wide representations of behavior spanning multiple timescales and states in C. elegans'. Each NWB file contains the NIR bright-field images, GCaMP images, and NeuroPAL structural images, ROI locations and IDs, labeled activity traces, and behavioral information.
```

To prototype, I want to just get the data for one of the worms, which is about 26GB. I downloaded the NWB file for one worm from this dataset and inspected it:
```
filepath = "sub-2022-06-14-07-SWF702_ses-20220614_behavior+image+ophys.nwb"
# Open the file in read mode "r",
io = NWBHDF5IO(filepath, mode="r")
nwbfile = io.read()
nwbfile
```




In [ ]:
# Install the required libraries
!pip install -U pynwb --quiet
!pip install "dandi>=0.60.0" --quiet

In [ ]:
from pynwb import NWBHDF5IO
from dandi.download import download
from dandi.dandiapi import DandiAPIClient
from fsspec.implementations.cached import CachingFileSystem
from IPython.display import HTML

import numpy as np
import h5py
import fsspec
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
# @title Download or use existing file

# download("https://api.dandiarchive.org/api/dandisets/000776/versions/0.241009.1509/assets/ece28dd8-5ca0-4c25-9a94-2f12b3b143e0/download/", ".")
# !dandi download https://api.dandiarchive.org/api/dandisets/000776/versions/0.241009.1509/assets/ece28dd8-5ca0-4c25-9a94-2f12b3b143e0/download/
filepath = "sub-2023-01-05-18-SWF702_ses-20230105_behavior+image+ophys.nwb" # change path of needed
io = NWBHDF5IO(filepath, mode="r")
nwbfile = io.read()
# The data we want is under `nwbfile.processing`
nwbfile

Data type,uint16
Shape,"(1615, 322, 212, 80, 2)"
Array size,32.86 GiB
Chunk shape,"(2, 41, 27, 20, 1)"
Compression,gzip
Compression opts,4
Compression ratio,2.0073965501638975
Data type,int64
Shape,"(2,)"
Array size,16.00 bytes
Chunk shape,None


In [ ]:
#@title .
# # DEBUG: Andrew test_worm.nwb file.
# filepath = "test_worm.nwb"
# io = NWBHDF5IO(filepath, mode="r")
# nwbfile = io.read()
# nwbfile
# dask_array = nwbfile.acquisition['raw_data'].data # shaped (T=11, X=1024, Y = 512, Z = 20, T = 2)
# t0, t1,z = 0, 0, 0
# frame = dask_array[t0, :, :, z, t1]

# # using matplotlib
# plt.imshow(frame, cmap='gray', vmin=0, vmax=255)
# plt.show()

# # Normalize the array to 0-255
# normalized_frame = (255 * (frame - frame.min()) / (frame.max() - frame.min())).astype(np.uint8)

# # Create and show the image
# from IPython.display import display
# image = Image.fromarray(normalized_frame)
# display(image)

To extract the calcium fluorescence signal of labeled neurons and the sequence of frames of the NIR behavior video from your NWB file, we will explore the two key components:

1. **Calcium fluorescence signal:** Access the `SignalCalciumImResponseSeries` under `nwbfile.processing['CalciumActivity']`. This includes the labeled neuron IDs and their corresponding fluorescence signals.

2. **NIR behavior video frames:** Access the `BrightFieldNIR` under `nwbfile.processing['BF_NIR']` to extract the video frames and timestamps.

In [ ]:
# 1. Extract calcium fluorescence signal (ordered by neuron IDs)
def extract_calcium_activity(nwbfile):
    """
    Extract calcium activity data including neuron IDs, fluorescence signals,
    and their associated timestamps.

    Parameters:
        nwbfile: The NWB file object.

    Returns:
        neuron_ids: List of neuron IDs.
        neural_activity: 2D numpy array of fluorescence signals (time x neurons).
        activity_timestamps: 1D numpy array of timestamps corresponding to neural_activity.
    """
    calcium_activity = nwbfile.processing['CalciumActivity']

    # Get neuron IDs
    neuron_ids = calcium_activity.data_interfaces['NeuronIDs'].labels[:]

    # Get calcium fluorescence signals
    neural_activity = calcium_activity.data_interfaces['SignalCalciumImResponseSeries'].data  # HDF5 dataset
    activity_timestamps = calcium_activity.data_interfaces['SignalCalciumImResponseSeries'].timestamps[:]

    return neuron_ids, neural_activity, activity_timestamps


# 2. Extract NIR behavior video frames
def extract_nir_video(nwbfile):
    """
    Extract NIR video data including frames and their associated timestamps.

    Parameters:
        nwbfile: The NWB file object.

    Returns:
        behavior_video: HDF5 dataset containing the NIR video frames.
        video_timestamps: 1D numpy array of timestamps corresponding to behavior_video.
    """
    bf_nir = nwbfile.processing['BF_NIR']

    # Get video data
    behavior_video = bf_nir.data_interfaces['BrightFieldNIR'].data  # HDF5 dataset
    video_timestamps = bf_nir.data_interfaces['BrightFieldNIR'].timestamps[:] / 1e9  # Convert nanoseconds to seconds

    return behavior_video, video_timestamps


# Extract calcium activity data
neuron_ids, neural_activity, activity_timestamps = extract_calcium_activity(nwbfile)
print(f"Extracted {len(neuron_ids)} neurons' calcium signals.")
print(f"Activity data shape: {neural_activity.shape}")
print(f"Activity timestamps shape: {activity_timestamps.shape}")
print(f"Activity timestamps range: {activity_timestamps.min()} to {activity_timestamps.max()}\n")

# Extract NIR video data
behavior_video, video_timestamps = extract_nir_video(nwbfile)
print(f"Extracted {behavior_video.shape[0]} frames of NIR video.")
print(f"Video data shape: {behavior_video.shape}")
print(f"Video timestamps shape: {video_timestamps.shape}")
print(f"Video timestamps range: {video_timestamps.min()} to {video_timestamps.max()}\n")

# Get the neural activtiy and behavior video to have the same number of timesteps
video_inds = np.unique(np.argmin(np.abs(activity_timestamps[:, np.newaxis] - video_timestamps[np.newaxis, :]), axis=1))

# debug: slicing is extremely slow. why?
behavior_video = behavior_video[video_inds]
video_timestamps = video_timestamps[video_inds]

# debug: we're not using these
neural_activity = neural_activity[:len(video_inds)]
activity_timestamps = activity_timestamps[:len(video_inds)]

# Save example data for validation
np.save("neural_activitiy.npy", neural_activity)  # save fluorescence traces
np.save("neuron_ids.npy", neuron_ids) # save neuron IDs
np.save("behavior_video.npy", behavior_video)  # save the corresponding video frames

# Debugging
print(neural_activity.shape, neuron_ids.shape, behavior_video.shape)
print(type(neural_activity), type(neuron_ids), type(behavior_video))

frames = behavior_video
fig, ax = plt.subplots()
def animate(i):
    ax.clear()
    ax.imshow(frames[i], cmap='gray')
    ax.set_title(f"Frame {i+1}/{len(frames)}")
    ax.axis('off')
    plt.close()
ani = animation.FuncAnimation(fig, animate, frames=len(frames), interval=50, repeat=True)
HTML(ani.to_html5_video())
# HTML(ani.to_jshtml())

Extracted 122 neurons' calcium signals.
Activity data shape: (1615, 122)
Activity timestamps shape: (1615,)
Activity timestamps range: 257796.84560312366 to 258767.78518050734

Extracted 19350 frames of NIR video.
Video data shape: (19350, 968, 732)
Video timestamps shape: (19350,)
Video timestamps range: 257786.454367864 to 258277.019770896

(800, 122) (122,) (800, 968, 732)
<class 'numpy.ndarray'> <class 'numpy.ndarray'> <class 'numpy.ndarray'>


---

# **Neural Activity and Neural ID Conditioned Behavior Diffusion**
## **Proposal for Neural-Sequence-to-Video-Diffusion**

---

### **Goal**
To build a pipeline leveraging diffusion models for generating behavior video sequences conditioned on neural activity time-series data and neuron IDs. This adapts the latent diffusion framework typically used for text-to-image generation.

---

### **Data Overview**
1. **Neural Activity Time Series**:
   - Shape: \(T \times N\), where \(T\) is the number of timesteps and \(N\) is the number of neurons.
   - Includes a corresponding length-\(T\) timestamp vector.

2. **Behavior Video**:
   - Shape: \(T \times H \times W\), where \(T\) is the number of frames (aligned to the neural activity), and \(H, W\) are frame dimensions.
   - Includes a corresponding length-\(T\) timestamp vector aligned with the neural activity.

3. **Neuron IDs**:
   - List of \(N\) neuron identifiers, treated as a sequence or "sentence."

---

### **Proposed Pipeline**

#### **1. Neural ID Encoder**
- A transformer-based text encoder (e.g., `CLIPTextModel`) to embed the neuron ID sequence into a latent vector.

#### **2. Neural Activity Encoder**
- A sequence model (e.g., LSTM or State Space Model) to process the calcium time-series data and output a vector representation.

#### **3. Combined Latent Representation**
- Concatenate the outputs of the Neural ID Encoder and Neural Activity Encoder to form a combined latent vector.

#### **4. Conditional UNet**
- A pretrained UNet2DConditionModel adapted from HuggingFace's diffusion pipeline, conditioned on the latent representation.
- This model generates latent video representations step by step in an autoregressive manner.

#### **5. Video Frame Decoder**
- A modified VAE decoder adapted to produce grayscale video frames.

#### **6. Loss Function**
- A frame-by-frame reconstruction loss (e.g., MSE) to train the model.

---

### **Plan**
1. **Data Preparation**:
   - Extract aligned neural activity and video sequences by interpolating timestamps if necessary.
   - Normalize neural activity and video frames to prepare them for input to the model.
   - Preprocess neuron IDs into tokenized sequences for the text encoder.

2. **Model Implementation**:
   - Use pretrained models for text encoding (e.g., `CLIPTextModel`) and diffusion (e.g., `UNet2DConditionModel`).
   - Implement a custom Neural Activity Encoder and a small feedforward module for autoregressive latent generation.
   - Adapt the VAE decoder to output grayscale frames.

3. **Training**:
   - Fine-tune the Neural ID Encoder, Neural Activity Encoder, and autoregressive feedforward module using the prepared dataset.
   - Freeze the pretrained UNet and optimize other components.

4. **Inference**:
   - Generate behavior videos for unseen neural activity data by conditioning on their latent representations.



In [ ]:
# Install required libraries
!pip install --quiet diffusers transformers accelerate torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.4 MB/s eta 0:00:00


C elegans neural activity-behavior diffusion

There seems to be still a long ways to go from the data to get a working diffusion model idea like what I envisioned. Much of it is perhaps because I don't understand a lot of the technical details. My challenge is conceptually mapping the idea from general text conditioned video diffusion to the scenario I have which can be summarized of "neural activity and neural ID conditioned behavior diffusion", to use similar words

Let me try to broadly describe the data I have and overall pipeline setup I would like.  I want you to write this all up neatly as a proposal in markdown and code cells of a Colab notebook describing and implementing in detail how I would, with the least amount of effort, use the image diffusion frameworks set up by existing pipelines (like HuggingFace Diffusers: https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb) and pretrained model (like https://huggingface.co/blog/stable_diffusion) to achieve that. I have attached a PDF of the Colab notebook we initially started working on but I am not sure if it is getting me to my goal.

Let's imagine we successfully processed out from the NWB for a worm downloaded from DANDI:
1a. T_f x N matrix of neural activity; N is the number of recorded neurons, T_f is the number of timesteps of recorded activity.
1b. A length T_f vector containing the timestamps (in seconds) corresponding to timesteps of the neural activity time series.
2a. The corresponding T_b x H x W tensor of grayscale frames of the worm behaving (an NIR video); the T_b is the number of timesteps of the behavior video which was sampled at a much faster rate than the calcium (the NIR camera captures every frame, the fluorescence signal is processed from volumes where there several frames per volume, hence why the number of timesteps T_b > T_f).
2b. A length T_b vector containing the timestamps (in seconds) corresponding to frames of the NIR behavior video.
3. I also have a length N list of the putative IDs of the N neurons in the neural activity. Some of these are the canonical neuron names if the neuron could have been identified from genetic markers, some are just '' if the neuron could not be ID'd and some have a '?' character in the neuron name when the dorsal/ventral or left-right-ness could not be disambiguated.

I imagine creating a dataset where each item in our dataset is (at minimum) the neural activity sequence, the neural ID sentence, and the behavior video tensor for one worm.

My overall vision of the pipeline is:
1. A simple text /sentence encoder (simple transformer-based encoder like the CLIPTextModel) that treats the array of neuron IDs like a sentence/sequence of input tokens and outputs
2. A neural activity series encoder that takes as input the matrix of neural activity (likely some sort of sequence model like LSTM or SSM) and outputs a vector representation.
3. We can somehow merge/combine the above two outputs to get the latent representation of the appropriate shape to give as the input a pretrained UNet2DConditionModel.
4. The latent output the conditional unet should be ran autoregressively for num_frames times (where num_frames = T_b) through a small custom residual feedforward network module that has the same output shape as the input shape (i.e. the shape of the latent) and we collect the length num_frames array of those outputs. Thus the conditional signal that guides the generation is based on the neural activity time series and the neuron IDs. The combination of above two modules together can be interpreted as the encoder part as their role is converting our inputs into the latent representation that we will use to condition the U-Net.
5. We run each of those through the vae.decoder to get the predicted num_frames of our model. We train the model with a loss that encourages the predicted video to match the true video frame by frame.

Notice how we are co-opting/adapting the setup from a text-to-image diffusion pipeline to do neural-sequence-to-video-diffusion. The "adapters" we have to define are the custom modules that work with the particular data we have; in particular, a text encoder that embeds the "sentence" of neural IDs into a vector that we concatenate with the hidden state output of encoding the neural time series with an LSTM/SSM model. We should use a pretrained text encoder for the former but the latter should be learnable module so that it works with the statistics of our neural data. The other "adapter" is that we will have a small feedforward module that we will use autotregressively generate num_frames latents since the UNet gives us just one denoised latent output for the latent input made from the neural activity and ID embedding.

I know generally speaking, diffusion models are machine learning systems that are trained to denoise random Gaussian noise step by step, to get to a sample of interest, I am trying to co-opt this essentially learn to map the neural signal (and IDs) into random Gaussian noise then is then diffused to the behavior (the video of the worm corresponding to those neural measurements). Also I am realizing that my whole agenda is to finetune the input and output part of the latent diffusion model pipeline to achieve my idea/work with my data so I am interested in training and inference (there seems to significant differences in how to use the models based on that). The difference is I am not training to denoise images. I am trying to "denoise" neural activity signals into video of behavior.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor, Normalize
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import StableDiffusionPipeline
import torch.nn as nn
import numpy as np

# Define Dataset
class NeuralBehaviorDataset(Dataset):
    def __init__(self, neural_activity, neuron_ids, behavior_video, tokenizer):
        """
        Dataset where each item is the full neural activity sequence, neuron IDs, and video tensor for one worm.

        Args:
            neural_activity (list of numpy.ndarray): List of neural activity arrays, one per worm (T x N).
            neuron_ids (list of list of str): List of neuron ID arrays, one per worm (length N).
            behavior_video (list of numpy.ndarray): List of behavior video tensors, one per worm (T x H x W).
            tokenizer (CLIPTokenizer): Tokenizer for encoding neuron IDs.
        """
        self.neural_activity = neural_activity
        self.neuron_ids = neuron_ids
        self.behavior_video = behavior_video
        self.tokenizer = tokenizer

        # Ensure consistency in temporal dimensions
        for act, vid in zip(self.neural_activity, self.behavior_video):
            assert act.shape[0] == vid.shape[0], "Temporal dimensions (T) of neural activity and video must match."

    def __len__(self):
        return len(self.neural_activity)

    def __getitem__(self, idx):
        neural_data = torch.tensor(self.neural_activity[idx], dtype=torch.float32)  # (T, N)
        video_tensor = torch.tensor(self.behavior_video[idx], dtype=torch.float32).unsqueeze(1)  # (T, 1, H, W)
        neuron_sentence = " ".join(self.neuron_ids[idx])  # Join neuron IDs into a sentence

        # Normalize video frames to [-1, 1]
        video_tensor = video_tensor / 255.0  # Scale to [0, 1]
        video_tensor = Normalize((0.5,), (0.5,))(video_tensor)  # Normalize to [-1, 1]

        # Tokenize neuron IDs
        neuron_tokens = self.tokenizer(neuron_sentence, return_tensors="pt", padding=True, truncation=True)

        return neural_data, neuron_tokens, video_tensor

# Load Pretrained Text Encoder
class NeuralIDEncoder(nn.Module):
    def __init__(self, pretrained_model="openai/clip-vit-base-patch32"):
        super().__init__()
        self.text_model = CLIPTextModel.from_pretrained(pretrained_model)

    def forward(self, tokenized_input):
        return self.text_model(**tokenized_input).pooler_output

# Neural Activity Encoder
class NeuralActivityEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, neural_data):
        output, _ = self.lstm(neural_data)  # (batch, T, N)
        return self.fc(output[:, -1, :])  # Final hidden state

# Combine Encoders
class CombinedEncoder(nn.Module):
    def __init__(self, text_encoder, activity_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.activity_encoder = activity_encoder

    def forward(self, neural_data, tokenized_text):
        text_embedding = self.text_encoder(tokenized_text)  # (batch, embedding_dim)
        activity_embedding = self.activity_encoder(neural_data)  # (batch, embedding_dim)
        return torch.cat([text_embedding, activity_embedding], dim=-1)

# Conditional UNet and Decoder Setup
class BehaviorDiffusionModel(nn.Module):
    def __init__(self, unet_model, vae_decoder, encoder_dim):
        super().__init__()
        self.unet = unet_model
        self.vae_decoder = vae_decoder

        # Replace fc_autoregressive with a 3-layer MLP
        self.fc_autoregressive = nn.Sequential(
            nn.Linear(encoder_dim, encoder_dim),
            nn.ReLU(),
            nn.Linear(encoder_dim, encoder_dim),
            nn.ReLU(),
            nn.Linear(encoder_dim, encoder_dim)
        )

    def forward(self, noise, num_steps):
        """
        Autoregressive generation of latent representations and their decoding to video frames.
        """
        generated_latent = []
        current_latent = self.unet(noise, timestep=torch.tensor([50], device="cuda"))
        for _ in range(num_steps):  # Autoregressive generation
            current_latent = self.fc_autoregressive(current_latent)
            generated_latent.append(current_latent)
        generated_latent = torch.stack(generated_latent, dim=1)

        # Decode the video frames
        decoded_frames = self.vae_decoder.decode(generated_latent)
        return decoded_frames

# Training Loop
class Trainer:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train(self, dataloader, epochs):
        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for neural_data, tokenized_text, video_tensor in dataloader:
                self.optimizer.zero_grad()

                # Encode inputs
                latent = self.model.unet.encoder(neural_data, tokenized_text)

                # Predict video frames
                predicted_frames = self.model(latent, video_tensor.size(0))

                # Compute loss
                loss = self.loss_fn(predicted_frames, video_tensor)
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader):.4f}")

# Example Usage
if __name__ == "__main__":
    num_worms = 10
    T, N, H, W = 100, 122, 64, 64
    batch_neural_activity = [np.random.rand(T, N) for _ in range(num_worms)]
    batch_neuron_ids = [[f"Neuron{i}" for i in range(N)] for _ in range(num_worms)]
    batch_behavior_video = [np.random.rand(T, H, W) for _ in range(num_worms)]

    tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    dataset = NeuralBehaviorDataset(batch_neural_activity, batch_neuron_ids, batch_behavior_video, tokenizer)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


    # Load the Stable Diffusion pipeline
    pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
    pipe.to("cuda")
    unet = pipe.unet  # UNet2DConditionModel
    vae_decoder = pipe.vae.decoder  # AutoencoderKL

    # Modify VAE Decoder to Generate Grayscale Images
    vae_decoder.conv_out = nn.Conv2d(
        in_channels=vae_decoder.conv_out.in_channels,
        out_channels=1,  # Grayscale
        kernel_size=vae_decoder.conv_out.kernel_size,
        stride=vae_decoder.conv_out.stride,
        padding=vae_decoder.conv_out.padding
    )

    # Freeze UNet Parameters
    for param in unet.parameters():
        param.requires_grad = False

    text_encoder = NeuralIDEncoder()
    activity_encoder = NeuralActivityEncoder(input_dim=N, hidden_dim=256, output_dim=512)
    combined_encoder = CombinedEncoder(text_encoder, activity_encoder)
    diffusion_model = BehaviorDiffusionModel(unet, vae_decoder, encoder_dim=1024)

    optimizer = torch.optim.Adam(
        list(combined_encoder.parameters()) + list(diffusion_model.fc_autoregressive.parameters()), lr=1e-4
    )
    loss_fn = nn.MSELoss()

    trainer = Trainer(diffusion_model, optimizer, loss_fn)
    trainer.train(dataloader, epochs=5)


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

scheduler/scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

AttributeError: 'UNet2DConditionModel' object has no attribute 'encoder'